In [29]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder, OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV, train_test_split

print("Done")

Done


In [2]:
df= pd.read_excel(r"D:\codes\databases\weather.xlsx")
df_test = pd.read_excel(r"D:\codes\databases\weather_test_150.xlsx")
df.columns

Index(['country', 'location_name', 'latitude', 'longitude', 'timezone',
       'last_updated_epoch', 'last_updated', 'temperature_celsius',
       'temperature_fahrenheit', 'condition_text', 'wind_mph', 'wind_kph',
       'wind_degree', 'wind_direction', 'pressure_mb', 'pressure_in',
       'precip_mm', 'precip_in', 'humidity', 'cloud', 'feels_like_celsius',
       'feels_like_fahrenheit', 'visibility_km', 'visibility_miles',
       'uv_index', 'gust_mph', 'gust_kph', 'air_quality_Carbon_Monoxide',
       'air_quality_Ozone', 'air_quality_Nitrogen_dioxide',
       'air_quality_Sulphur_dioxide', 'air_quality_PM2.5', 'air_quality_PM10',
       'air_quality_us-epa-index', 'air_quality_gb-defra-index', 'sunrise',
       'sunset', 'moonrise', 'moonset', 'moon_phase', 'moon_illumination'],
      dtype='object')

In [3]:
features_to_drop = [
    'latitude', 'longitude', 'timezone', 'condition_text', 'last_updated_epoch',
    'last_updated', 'country', 'location_name', 'sunrise', 'sunset',
    'moonrise', 'moonset', 'moon_illumination', 'moon_phase'
]
X = df.drop(columns=features_to_drop)
y = df['condition_text']

In [4]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [5]:
X_train, X_val, y_train, y_val = train_test_split(X, y_encoded, test_size=0.2, random_state=42)


In [6]:
numerical_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(include='object').columns.tolist()
# Made separate lists of categorica and numeric columns

In [7]:
X_train = X_train.dropna()
# Dropped columns with missing values

In [8]:
#numerical_features.remove("moon_illumination")
#categorical_features.remove("moon_phase")
#X_train = X_train.drop(columns=["moon_illumination", "moon_phase"])
#X_val = X_val.drop(columns=["moon_illumination", "moon_phase"])

In [9]:
categorical_features
numerical_features

['temperature_celsius',
 'temperature_fahrenheit',
 'wind_mph',
 'wind_kph',
 'wind_degree',
 'pressure_mb',
 'pressure_in',
 'precip_mm',
 'precip_in',
 'humidity',
 'cloud',
 'feels_like_celsius',
 'feels_like_fahrenheit',
 'visibility_km',
 'visibility_miles',
 'uv_index',
 'gust_mph',
 'gust_kph',
 'air_quality_Carbon_Monoxide',
 'air_quality_Ozone',
 'air_quality_Nitrogen_dioxide',
 'air_quality_Sulphur_dioxide',
 'air_quality_PM2.5',
 'air_quality_PM10',
 'air_quality_us-epa-index',
 'air_quality_gb-defra-index']

In [10]:
numerical_preprocessor = Pipeline(steps= [
    ("scaler", StandardScaler())
])

categorical_preprocessor = Pipeline(steps= [
    ("encoder", OneHotEncoder())
])

In [11]:
condition_text_transform = ColumnTransformer(transformers= [
    ("conditional_transform", OrdinalEncoder(), ["condition_text"])
])

In [12]:
preprocessor = ColumnTransformer(transformers=[
    ("numeric_transformers", numerical_preprocessor, numerical_features),
    ("categorical_transform", categorical_preprocessor, categorical_features),
], remainder="passthrough")


# Built a preprocessor transformer, where all preprocessing tasks take place at once.

In [13]:
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42)) # Using the classifier
])
# Built a pipeline that has two steps: the preprocessor and the model.

In [18]:
param_grid = {
    "classifier__n_estimators": [200],
    "classifier__max_depth": [None],
    "classifier__min_samples_leaf": [1]
}

In [15]:
grid_search = GridSearchCV(pipeline, param_grid=param_grid, cv=3, n_jobs=-1, verbose=2, scoring='accuracy')

grid_search.fit(X_train, y_train)

print("\nBest parameters found:", grid_search.best_params_)
print("Best accuracy score:", grid_search.best_score_)

Fitting 3 folds for each of 27 candidates, totalling 81 fits


C:\Users\Shreshth Kabra\AppData\Roaming\Python\Python313\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(



Best parameters found: {'classifier__max_depth': None, 'classifier__min_samples_leaf': 1, 'classifier__n_estimators': 200}
Best accuracy score: 0.8484288815258395


In [19]:
y_pred = grid_search.best_estimator_.predict(X_val)
accuracy = accuracy_score(y_val, y_pred)
print("Validation accuracy:", accuracy)

Validation accuracy: 0.8559357087986763


In [ ]:
final_pred= grid_search.best_estimator_.predict(df_test)
final_pred = le.inverse_transform(final_pred)

In [24]:
y = df[['latitude', 'longitude']]

In [25]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [26]:
numerical_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(include='object').columns.tolist()


In [27]:
numerical_preprocessor = Pipeline(steps=[
    ("scaler", StandardScaler())
])

# Pipeline for one-hot encoding categorical data
categorical_preprocessor = Pipeline(steps=[
    ("encoder", OneHotEncoder(handle_unknown='ignore'))
])

In [28]:
preprocessor = ColumnTransformer(transformers=[
    ("numeric_transformers", numerical_preprocessor, numerical_features),
    ("categorical_transform", categorical_preprocessor, categorical_features),
], remainder="passthrough")

In [30]:
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(random_state=42)) # Using the regressor
])

In [33]:
param_grid = {
    "regressor__n_estimators": [50, 100, 200],
    "regressor__max_depth": [10, 20, 30, None],
    "regressor__min_samples_leaf": [2, 4, 1]
}

In [34]:
grid_search = GridSearchCV(pipeline, param_grid=param_grid, cv=3, n_jobs=-1, verbose=2, scoring='neg_mean_squared_error')

grid_search.fit(X_train, y_train)

print("\nBest parameters found:", grid_search.best_params_)
print("Best neg_mean_squared_error score:", grid_search.best_score_)

Fitting 3 folds for each of 36 candidates, totalling 108 fits

Best parameters found: {'regressor__max_depth': 30, 'regressor__min_samples_leaf': 1, 'regressor__n_estimators': 200}
Best neg_mean_squared_error score: -500.0612845366418


In [35]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_val)
Validation_mse = np.mean((y_val - y_pred) ** 2)
print("Validation MSE:", Validation_mse)

Validation MSE: 467.78677346857006


In [36]:
final_pred_latlong = best_model.predict(df_test)
final_pred_latlong = pd.DataFrame(final_pred_latlong, columns=['latitude', 'longitude'])


In [43]:
pd.DataFrame(final_pred).to_csv(r"D:\codes\databases\final_predictions.csv", index=False)